# Práctica: Wireframe y gráficos para un dashboard de FIFA
#### Objetivo: diseñar cinco elementos visuales utilizando pandas, matplotlib y seaborn, con el archivo fifa_eda.csv.

## Preparación: importar librerías y localizar el archivo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Carpeta donde guardo los archivos en mi Mac
carpeta = Path('/Users/santiago/Downloads')

## Paso 1. Leer el archivo y revisar las columnas
#### Primero leo los datos con pandas, después muestro las primeras filas y los nombres de las columnas.

In [ ]:
archivo = carpeta / 'fifa_eda.csv'

# A veces el navegador agrega (1) cuando descargamos un archivo repetido
if not archivo.exists():
    archivo = carpeta / 'fifa_eda(1).csv'

df = pd.read_csv(archivo)

print('Número de filas y columnas:', df.shape)
print('Columnas:', df.columns.tolist())
display(df.head())

## Paso 2. Preparar los datos
#### Compruebo que las columnas que voy a utilizar sean numéricas. La altura del CSV está expresada en pies decimales, por lo que creo otra columna con su equivalente aproximado en centímetros. No cambio el archivo original.

In [ ]:
columnas_numericas = [
    'Age', 'Overall', 'Potential', 'Value', 'Wage',
    'International Reputation', 'Skill Moves',
    'Height', 'Weight', 'Release Clause'
]

for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

# Transformo pies a centímetros para que sea más fácil interpretar la altura
df['Height_cm'] = df['Height'] * 30.48

print('Jugadores:', len(df))
print('Altura (ejemplo):', round(df.loc[0, 'Height_cm'], 2), 'cm')

## Grafica 1. Matriz de correlación y heatmap
#### Una matriz de correlación permite ver qué tan relacionadas están dos variables numéricas. Los valores se encuentran entre -1 y 1. Para evitar resultados poco útiles, no utilizo la columna ID porque funciona como identificador de cada jugador.

In [ ]:
columnas_correlacion = [
    'Age', 'Overall', 'Potential', 'Value', 'Wage',
    'International Reputation', 'Skill Moves',
    'Height_cm', 'Weight', 'Release Clause'
]

matriz = df[columnas_correlacion].corr()

plt.figure(figsize=(11, 7))
sns.heatmap(matriz, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Grafica 1. Correlación entre características de jugadores')
plt.tight_layout()
plt.savefig(carpeta / '01_heatmap_fifa.png', dpi=140, bbox_inches='tight')
plt.show()

## Grafica 2. ¿Cuál es la relación entre la edad y el Overall?
#### Utilizo un gráfico de dispersión con todos los jugadores y una línea con el promedio de Overall para las edades que tengan al menos 50 jugadores (evitando promedios poco representativos). También calculo la correlación de Pearson.

In [ ]:
relacion = df[['Age', 'Overall']].dropna()
correlacion = relacion['Age'].corr(relacion['Overall'])
# Para la línea de promedios uso edades con suficientes jugadores
promedios_edad = relacion.groupby('Age')['Overall'].agg(['mean', 'count'])
promedios_edad = promedios_edad[promedios_edad['count'] >= 50]['mean']

plt.figure(figsize=(10, 5))
plt.scatter(relacion['Age'], relacion['Overall'], alpha=0.12, s=12,
            color='#247B9B', label='Jugadores')
plt.plot(promedios_edad.index, promedios_edad.values,
         color='#E07A35', linewidth=2.5, label='Overall promedio (mín. 50 jugadores)')
plt.title('Grafica 2. Edad vs Overall')
plt.xlabel('Edad (años)')
plt.ylabel('Overall (valoración general)')
plt.legend()
plt.tight_layout()
plt.savefig(carpeta / '02_edad_vs_overall.png', dpi=140, bbox_inches='tight')
plt.show()

print('Correlación entre edad y Overall:', round(correlacion, 2))

## Grafica 3. Cantidad de jugadores por club
#### Como hay muchos clubes, para que la gráfica sea fácil de leer mostraré los 15 clubes con más jugadores registrados. Los clubes sin nombre se omiten de este conteo.

In [ ]:
clubes = df['Club'].dropna().value_counts().head(15)

plt.figure(figsize=(10, 6))
plt.barh(clubes.index[::-1], clubes.values[::-1], color='#2B7A92')
plt.title('Grafica 3. Los 15 clubes con más jugadores registrados')
plt.xlabel('Número de jugadores')
plt.ylabel('Club')
plt.tight_layout()
plt.savefig(carpeta / '03_jugadores_por_club.png', dpi=140, bbox_inches='tight')
plt.show()

print('Clubes analizados:', df['Club'].nunique())

## Grafica 4. Altura vs Skill Moves, según el pie preferido
#### Creo un multipanel con dos gráficos: uno para jugadores zurdos y otro para diestros. Ambos usan la misma escala para comparar mejor los resultados.

In [ ]:
datos_altura = df[['Height_cm', 'Skill Moves', 'Preferred Foot']].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)

for ax, pie, color in zip(axes, ['Left', 'Right'], ['#2B7A92', '#E0A454']):
    parte = datos_altura[datos_altura['Preferred Foot'] == pie]
    ax.scatter(parte['Height_cm'], parte['Skill Moves'],
               s=13, alpha=0.16, color=color)
    ax.set_title('Zurdos' if pie == 'Left' else 'Diestros')
    ax.set_xlabel('Altura aproximada (cm)')
    ax.set_ylim(0.5, 5.5)
    ax.grid(alpha=0.25)
    print(('Zurdos' if pie == 'Left' else 'Diestros') + ':', len(parte))

axes[0].set_ylabel('Skill Moves (1 a 5)')
fig.suptitle('Grafica 4. Altura vs Skill Moves por pie preferido')
fig.tight_layout()
fig.savefig(carpeta / '04_altura_skill_moves_pie.png', dpi=140, bbox_inches='tight')
plt.show()

## Grafica 5. Jóvenes con mayor potencial
#### Para visualizar a los posibles futuros cracks, elijo jugadores de 23 años o menos con Potential de al menos 85. Luego muestro los 12 con mayor potencial. Es una selección creada para la práctica, no una predicción real del futuro.


In [ ]:
jovenes = df[(df['Age'] <= 23) & (df['Potential'] >= 85)].copy()
top_jovenes = jovenes.nlargest(12, 'Potential').sort_values('Potential')

# Etiquetas con nombre y edad para reconocerlos fácilmente
etiquetas = top_jovenes['Name'] + ' (' + top_jovenes['Age'].astype(int).astype(str) + ' años)'

plt.figure(figsize=(10, 6))
plt.barh(etiquetas, top_jovenes['Potential'], color='#2B7A92')
plt.title('Grafica 5. Jóvenes con mayor potencial')
plt.xlabel('Potential')
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig(carpeta / '05_jovenes_promesas.png', dpi=140, bbox_inches='tight')
plt.show()

display(top_jovenes[['Name', 'Age', 'Overall', 'Potential', 'Club']].sort_values('Potential', ascending=False))